In [1]:
import torch
import sys, os

# 确保能 import model 目录
sys.path.insert(0, os.path.abspath('.'))

print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available else 'cpu'
print(f'\n使用设备: {device}')

/Users/kevinpan/.pyenv/versions/LLM/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PyTorch 版本: 2.6.0
CUDA 可用: False

使用设备: mps


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('./model')
print(f'Tokenizer 加载成功，词表大小: {tokenizer.vocab_size}')
print(f'bos_token_id: {tokenizer.bos_token_id}, eos_token_id: {tokenizer.eos_token_id}')

/Users/kevinpan/.pyenv/versions/LLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer 加载成功，词表大小: 6400
bos_token_id: 1, eos_token_id: 2


In [4]:
from model.model_minimind import MiniMindConfig
from model.model_minimind import MiniMindForCausalLM

WEIGHT_PATH = './out/full_sft_768.pth'

config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=16,    
    num_attention_heads=8,
    num_key_value_heads=2,
    vocab_size=6400,
    use_moe=False,    
)

model = MiniMindForCausalLM(config)

# 加载 .pth state dict
state_dict = torch.load(WEIGHT_PATH, map_location=device)

# 兼容带/不带 'model.' 前缀的权重
if all(k.startswith('model.') for k in state_dict.keys()):
    state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'Missing keys  : {missing}')
print(f'Unexpected keys: {unexpected}')

model = model.to(device).eval()

total = sum(p.numel() for p in model.parameters())
print(f'\n模型加载成功 ✓  参数量: {total/1e6:.1f}M')
print(f'已加载权重: {WEIGHT_PATH}')

ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?

## Cell 4 — 推理函数

In [ ]:
def generate(
    prompt: str,
    mode: str = 'sft',          # 'sft' | 'pretrain'
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
):
    """
    mode='sft'     : 使用 chat template，适合 full_sft 权重
    mode='pretrain': 直接续写 prompt，适合 pretrain 权重
    """
    if mode == 'sft':
        messages = [{'role': 'user', 'content': prompt}]
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt'
        ).to(device)
    else:
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = input_ids.clone()
        past_key_values = None

        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=input_ids if past_key_values is None else input_ids[:, -1:],
                past_key_values=past_key_values,
                use_cache=True
            )
            logits = outputs.logits[:, -1, :]  # [1, vocab_size]
            past_key_values = outputs.past_key_values

            # Repetition penalty
            if repetition_penalty != 1.0:
                for token_id in set(output_ids[0].tolist()):
                    logits[0, token_id] /= repetition_penalty

            # Temperature + Top-p sampling
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_probs[cumulative - sorted_probs > top_p] = 0
            sorted_probs /= sorted_probs.sum()
            next_token = sorted_idx[0, torch.multinomial(sorted_probs[0], 1)]

            if next_token.item() == tokenizer.eos_token_id:
                break

            output_ids = torch.cat([output_ids, next_token.unsqueeze(0).unsqueeze(0)], dim=1)
            input_ids = next_token.unsqueeze(0).unsqueeze(0)

    # 只解码新生成的部分
    new_tokens = output_ids[0, input_ids.shape[1] if mode == 'pretrain' else 
                            tokenizer.apply_chat_template([{'role':'user','content':prompt}], 
                            add_generation_prompt=True, return_tensors='pt').shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# 更简洁的包装（使用 model.generate）
def chat(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7):
    """直接使用 HuggingFace generate 接口（推荐）"""
    messages = [{'role': 'user', 'content': prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        output[0][input_ids.shape[1]:], skip_special_tokens=True
    )
    return response


print('推理函数已定义 ✓')
print('  chat(prompt)      → SFT 指令对话')
print('  generate(prompt, mode="pretrain") → 续写')

## Cell 5 — SFT 效果测试（用于 README 对比表格）

In [ ]:
# README 对比表格所需的 4 组 prompt
test_prompts = [
    '介绍一下你自己',
    '中国的首都是哪里？',
    '用一句话解释机器学习',
    '写一首关于春天的诗',
]

print('=' * 60)
print('SFT 模型输出（用于填写 README 对比表格）')
print('=' * 60)

for prompt in test_prompts:
    print(f'\n【Prompt】{prompt}')
    response = chat(prompt, max_new_tokens=150)
    print(f'【输出】{response}')
    print('-' * 40)

## Cell 6 — Pretrain vs SFT 对比（完整版）

In [ ]:
# 先用 pretrain 权重跑一遍，再换 sft，用于生成 README 对比数据
# 这里演示切换方式

def load_weights(weight_path: str):
    """动态切换权重"""
    state_dict = torch.load(weight_path, map_location=device)
    if all(k.startswith('model.') for k in state_dict.keys()):
        state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    print(f'已切换权重: {weight_path}')


def pretrain_generate(prompt: str, max_new_tokens: int = 100):
    """Pretrain 续写模式"""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)


test_prompt = '介绍一下你自己'

# Pretrain 输出
load_weights('./out/pretrain_768.pth')
pt_output = pretrain_generate(test_prompt)
print(f'[Pretrain] {test_prompt}')
print(f'输出: {pt_output[:200]}')

print()

# SFT 输出
load_weights('./out/full_sft_768.pth')
sft_output = chat(test_prompt)
print(f'[SFT] {test_prompt}')
print(f'输出: {sft_output}')

## Cell 7 — 自由对话

In [ ]:
# 确保加载的是 SFT 权重
# load_weights('./out/full_sft_768.pth')

# ↓ 修改这里输入你的问题
your_question = '请介绍一下 Transformer 架构的核心组件'

response = chat(your_question, max_new_tokens=300, temperature=0.7)
print(f'Q: {your_question}')
print(f'A: {response}')

## Cell 8 — 参数规模验证

In [ ]:
# 打印模型结构和参数量（用于确认 hidden_size 是否正确）
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'总参数量:     {total:,}  ({total/1e6:.1f}M)')
print(f'可训练参数量: {trainable:,}  ({trainable/1e6:.1f}M)')
print(f'\nhidden_size: {model.config.hidden_size}')
print(f'num_layers:  {model.config.num_hidden_layers}')
print(f'num_heads:   {model.config.num_attention_heads}')
print(f'use_moe:     {model.config.use_moe}')

# 如果参数量不对，说明 config 的 num_hidden_layers 需要调整
# 常见配置：hidden_size=768, layers=16 → 约 110M
#           hidden_size=768, layers=12 → 约 85M